In [ ]:
import pandas as pd
from collections import Counter
from nltk.util import bigrams, trigrams, ngrams

# Baca dataset
df = pd.read_csv("../pertemuan-03/scrape_kompas_clean.csv")
text_column = "processed_text"
df = df.dropna(subset=[text_column])

# Gabung jadi korpus token
corpus_tokens = []
for text in df[text_column]:
    tokens = text.split()
    corpus_tokens.extend(tokens)

# 1. Unigram
unigram = Counter(corpus_tokens)
print("Jumlah unigram:", len(unigram))

# 2. Bigram
bigram = Counter(bigrams(corpus_tokens))
print("Jumlah bigram:", len(bigram))

# 3. Trigram
trigram = Counter(trigrams(corpus_tokens))
print("Jumlah trigram:", len(trigram))

Jumlah unigram: 25
Jumlah bigram: 31
Jumlah trigram: 32


In [ ]:
total_unigram = sum(unigram.values())

# 5 kata yang paling sering muncul
top5_words = unigram.most_common(5)
print("Probabilitas 5 kata teratas:")
for kata, jumlah in top5_words:
    prob = jumlah / total_unigram
    print(f"- {kata}: {jumlah}/{total_unigram} = {prob:.4f}")

Probabilitas 5 kata teratas:
- aktor: 5/35 = 0.1429
- tinggal: 4/35 = 0.1143
- dunia: 3/35 = 0.0857
- temu: 2/35 = 0.0571
- deret: 1/35 = 0.0286


In [ ]:
kalimat_uji = [
    "aktor thailand",
    "tinggal di usia",
    "film baru ini",
    "pemeran utama",
    "cerita tentang"
]

print("=== Probabilitas Kalimat ===")
for kalimat in kalimat_uji:
    tokens = kalimat.split()
    prob = 1.0
    for kata in tokens:
        prob *= unigram[kata] / total_unigram if unigram[kata] > 0 else 1e-10
    print(f"Kalimat: {kalimat}")
    print(f"Probabilitas: {prob:.8f}\n")

=== Probabilitas Kalimat ===
Kalimat: aktor thailand
Probabilitas: 0.00408163

Kalimat: tinggal di usia
Probabilitas: 0.00000000

Kalimat: film baru ini
Probabilitas: 0.00000000

Kalimat: pemeran utama
Probabilitas: 0.00000000

Kalimat: cerita tentang
Probabilitas: 0.00000000



In [ ]:
import math

def perplexity(kalimat, model, N):
    tokens = kalimat.split()
    n = len(tokens)
    if n < 2: return 0
    prob_log = 0
    for w in model.keys():
        prob_log += math.log(model[w]/total_unigram if total_unigram > 0 else 1e-10)
    return math.exp(-prob_log / n)

# Contoh untuk unigram
for kalimat in kalimat_uji:
    pp = perplexity(kalimat, unigram, 1)
    print(f"Perplexity: {kalimat} → {pp:.4f}")

Perplexity: aktor thailand → 1824987656288075520.0000
Perplexity: tinggal di usia → 1493390167391.8086
Perplexity: film baru ini → 1493390167391.8086
Perplexity: pemeran utama → 1824987656288075520.0000
Perplexity: cerita tentang → 1824987656288075520.0000


In [5]:
def prediksi_kata(konteks, model_bigram, n=3):
    kandidat = {k:v for k,v in model_bigram.items() if k[0]==konteks}
    if not kandidat:
        return ["(tidak ada prediksi)"]
    urut = sorted(kandidat.items(), key=lambda x: -x[1])[:n]
    return [k[1] for k,_ in urut]

konteks_list = ["aktor", "thailand", "tinggal", "usia", "film"]
for kata in konteks_list:
    pred = prediksi_kata(kata, bigram)
    print(f"Setelah '{kata}': {pred}")

Setelah 'aktor': ['peran', 'ryotaro', 'marvelous']
Setelah 'thailand': ['jorjay']
Setelah 'tinggal': ['dunia', 'usia']
Setelah 'usia': ['(tidak ada prediksi)']
Setelah 'film': ['(tidak ada prediksi)']


In [6]:
V = len(unigram)
total_vocab = total_unigram + V

def prob_laplace(kata):
    return (unigram[kata] + 1) / total_vocab

# Cek 5 kata
for kata, _ in top5_words:
    print(f"Laplace {kata}: {prob_laplace(kata):.4f}")

Laplace aktor: 0.1000
Laplace tinggal: 0.0833
Laplace dunia: 0.0667
Laplace temu: 0.0500
Laplace deret: 0.0333
